# Train the "is this actually a scan" sanity gate

Trains two small binary classifiers (resnet18, ImageNet-pretrained) that reject uploads which aren't real chest X-rays / CT scans at all (random photos, scanned documents, screenshots) — something the existing modality gate and OOD detector in this project were never calibrated to catch.

**Before running:** upload `sanity_export.zip` (given to you alongside this notebook) using the cell below. It contains two ImageFolder-shaped datasets:

```
xray_sanity/{train,val,test}/{scan,not_scan}/*.jpg
ct_sanity/{train,val,test}/{scan,not_scan}/*.jpg
```

`scan` = real X-ray/CT images from this project's own dataset. `not_scan` = a mix of synthetic scanned-document images, synthetic textures/photos, and non-medical images from this repo (charts, screenshots, diagrams). Images are already resized to 224x224 (the model's actual input size), so the zip is small — nothing is lost by that, the training pipeline resizes to the same size anyway.

**After running:** download `xray_sanity_best.pt` and `ct_sanity_best.pt` from the file browser (or via the download cell at the end) and send both files back.

## 1. Upload and unzip the dataset

In [ ]:
# Run this in Google Colab. If you're not on Colab, skip this cell and
# instead place sanity_export.zip next to this notebook, then run the
# unzip cell below with ZIP_PATH pointing at it.
try:
    from google.colab import files
    uploaded = files.upload()  # pick sanity_export.zip in the dialog
    ZIP_PATH = next(iter(uploaded.keys()))
except ImportError:
    ZIP_PATH = "sanity_export.zip"  # adjust if running locally / elsewhere
print("ZIP_PATH =", ZIP_PATH)

In [ ]:
import zipfile
from pathlib import Path

DATA_ROOT = Path("sanity_data")
DATA_ROOT.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(DATA_ROOT)

XRAY_DIR = DATA_ROOT / "xray_sanity"
CT_DIR = DATA_ROOT / "ct_sanity"
assert (XRAY_DIR / "train").exists(), f"expected {XRAY_DIR/'train'} to exist after unzip"
assert (CT_DIR / "train").exists(), f"expected {CT_DIR/'train'} to exist after unzip"
print("OK:", XRAY_DIR, CT_DIR)

## 2. Model, transforms, dataset helpers

Copied 1:1 from this project's `ml/model.py` and `ml/dataset.py` (resnet18 path only) so the checkpoint this notebook produces is loadable by the backend's existing `models.py` without any changes there.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224


def build_transforms(train: bool) -> transforms.Compose:
    # Same pipeline the backend uses at inference time (gradcam_utils.py /
    # ml/dataset.py) -- must match exactly, including the Grayscale step
    # (everything the model ever sees is grayscale-3channel, so training
    # on that is what the served model needs to be good at).
    ops = [
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    ]
    if train:
        ops += [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
        ]
    ops += [
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    return transforms.Compose(ops)


class ImageClassifier(nn.Module):
    def __init__(self, num_classes: int = 2, pretrained: bool = True):
        super().__init__()
        self.backbone_name = "resnet18"
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)
        self._head_names = {"fc"}

    def forward(self, x):
        return self.backbone(x)

    def forward_features(self, x):
        out = self.backbone.conv1(x)
        out = self.backbone.bn1(out)
        out = self.backbone.relu(out)
        out = self.backbone.maxpool(out)
        out = self.backbone.layer1(out)
        out = self.backbone.layer2(out)
        out = self.backbone.layer3(out)
        out = self.backbone.layer4(out)
        out = self.backbone.avgpool(out)
        return torch.flatten(out, 1)

    def _is_head_param(self, name):
        return name.split(".")[0] in self._head_names

    def freeze_backbone(self):
        for name, p in self.backbone.named_parameters():
            p.requires_grad = self._is_head_param(name)

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True


def build_datasets(data_dir: Path):
    train_ds = datasets.ImageFolder(data_dir / "train", transform=build_transforms(train=True))
    val_ds = datasets.ImageFolder(data_dir / "val", transform=build_transforms(train=False))
    test_ds = datasets.ImageFolder(data_dir / "test", transform=build_transforms(train=False))
    assert train_ds.classes == val_ds.classes == test_ds.classes, (
        f"class mismatch: {train_ds.classes} {val_ds.classes} {test_ds.classes}"
    )
    return train_ds, val_ds, test_ds


def compute_class_weights(train_ds):
    num_classes = len(train_ds.classes)
    counts = torch.zeros(num_classes, dtype=torch.long)
    for _, label in train_ds.samples:
        counts[label] += 1
    total = counts.sum().item()
    weights = total / (num_classes * counts.float())
    return weights, counts

## 3. Training loop

Same recipe as `ml/train.py`: a few frozen-backbone warmup epochs (only the new head trains), then unfreeze and fine-tune the whole network at a lower LR. Weighted cross-entropy handles the scan/not_scan class imbalance (this matters most for CT, whose positive class is much smaller than X-ray's).

In [ ]:
def run_one_epoch(model, loader, criterion, optimizer, train: bool):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total


def train_sanity_gate(data_dir: Path, out_path: Path, epochs=6, warmup_epochs=1,
                       batch_size=32, lr=1e-3, finetune_lr=1e-4, num_workers=2):
    train_ds, val_ds, _ = build_datasets(data_dir)
    class_names = train_ds.classes
    print(f"classes: {class_names}")

    class_weights, class_counts = compute_class_weights(train_ds)
    for name, cnt, w in zip(class_names, class_counts.tolist(), class_weights.tolist()):
        print(f"  {name}: count={cnt} weight={w:.4f}")

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)

    model = ImageClassifier(num_classes=len(class_names), pretrained=True).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

    best_val_acc, best_epoch = -1.0, -1
    for epoch in range(1, epochs + 1):
        if epoch == 1:
            model.freeze_backbone()
            optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
            print(f"epoch {epoch}: backbone FROZEN, lr={lr}")
        elif epoch == warmup_epochs + 1:
            model.unfreeze_backbone()
            optimizer = optim.Adam(model.parameters(), lr=finetune_lr)
            print(f"epoch {epoch}: backbone UNFROZEN, lr={finetune_lr}")

        train_loss, train_acc = run_one_epoch(model, train_loader, criterion, optimizer, train=True)
        val_loss, val_acc = run_one_epoch(model, val_loader, criterion, optimizer, train=False)
        print(f"epoch {epoch}/{epochs}  train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc, best_epoch = val_acc, epoch
            torch.save({
                "model_state_dict": model.state_dict(),
                "backbone": "resnet18",
                "num_classes": len(class_names),
                "class_names": class_names,
                "val_acc": val_acc,
                "epoch": epoch,
            }, out_path)

    print(f"best val_acc={best_val_acc:.4f} at epoch {best_epoch} -> saved to {out_path}")
    return best_val_acc

## 4. Train both gates

X-ray has far more data (15k+ train images) than CT (~830), so CT gets a few more epochs to make sure it actually converges — bump either if val_acc is still climbing at the last epoch printed above.

In [ ]:
xray_val_acc = train_sanity_gate(
    XRAY_DIR, Path("xray_sanity_best.pt"),
    epochs=6, warmup_epochs=1, batch_size=32,
)

In [ ]:
ct_val_acc = train_sanity_gate(
    CT_DIR, Path("ct_sanity_best.pt"),
    epochs=8, warmup_epochs=1, batch_size=32,
)

## 5. Sanity-check on a couple of adversarial examples

Quick gut check before downloading: a plain grayscale synthetic "document" (blank page + black text-block rectangles) should be classified `not_scan` with high confidence. This is the exact failure mode that motivated training this gate — the existing pipeline gave a *real* black-and-white document 97% confidence "Tuberculosis".

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

def make_fake_document():
    img = Image.new("RGB", (300, 400), (250, 250, 250))
    draw = ImageDraw.Draw(img)
    y = 20
    while y < 380:
        draw.rectangle([20, y, 220, y + 10], fill=(20, 20, 20))
        y += 22
    return img

def check(ckpt_path, img):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = ImageClassifier(num_classes=len(ckpt["class_names"]), pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    x = build_transforms(train=False)(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0].tolist()
    return dict(zip(ckpt["class_names"], probs))

fake_doc = make_fake_document()
print("xray_sanity on fake document:", check("xray_sanity_best.pt", fake_doc))
print("ct_sanity on fake document:  ", check("ct_sanity_best.pt", fake_doc))

## 6. Download the checkpoints

Grab both `.pt` files (via the Colab file browser on the left, or the cell below) and send them back.

In [ ]:
try:
    from google.colab import files
    files.download("xray_sanity_best.pt")
    files.download("ct_sanity_best.pt")
except ImportError:
    print("Not on Colab -- files are already on disk at xray_sanity_best.pt / ct_sanity_best.pt")